# Model 2 V3: Fine-tuned DistilBERT — Mean Pooling

**Architecture:** DistilBERT mean pooling (avg all token embeddings) + regression head  
**Change from V2:** Mean pooling replaces [CLS] token pooling  
**Hypothesis:** Mean pooling captures more distributed information than [CLS], which was pretrained for classification  

```
V1/V2: DistilBERT → [CLS] token (768-dim) → head → price
V3:    DistilBERT → mean(all tokens) (768-dim) → head → price
```

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [1]:
from pricer.items import Item
from pricer.distilbert_model_v3 import DistilBERTRunnerV3
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 800,000 | Val: 10,000 | Test: 10,000


## 2. Setup Model

DistilBERT with mean pooling head.

In [3]:
runner = DistilBERTRunnerV3(train, val[:1000])
runner.setup(batch_size=128)

Loading DistilBERT tokenizer...
Loading DistilBERT model...
DistilBERT Regressor: 66,561,537 params (encoder: 66,362,880, head: 198,657)
Using cuda
DistilBERT V3 (mean pooling): 66,561,537 params (encoder: 66,362,880, head: 198,657)


## 3. Train

Max 15 epochs, early stopping patience=3.

In [4]:
history = runner.train(epochs=15, patience=3, warmup_steps=1000)

Epoch 1/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [1/15]
  Train Loss: 0.4730, Val Loss: 0.4092
  Val MAE: $56.15, LR: 0.00001887
  ** New best Val MAE: $56.15


Epoch 2/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [2/15]
  Train Loss: 0.3874, Val Loss: 0.3818
  Val MAE: $53.93, LR: 0.00001752
  ** New best Val MAE: $53.93


Epoch 3/15:   0%|          | 0/6250 [00:00<?, ?it/s]

...

Epoch [11/15]
  Train Loss: 0.2315, Val Loss: 0.3377
  Val MAE: $46.27, LR: 0.00000539
  No improvement (1/3)


Epoch 12/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [12/15]
  Train Loss: 0.2237, Val Loss: 0.3378
  Val MAE: $46.20, LR: 0.00000404
  No improvement (2/3)


Epoch 13/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [13/15]
  Train Loss: 0.2179, Val Loss: 0.3373
  Val MAE: $46.34, LR: 0.00000270
  No improvement (3/3)
Early stopping at epoch 13. Best Val MAE: $45.50


## 4. Training History

In [5]:
plot_training_history(history, title="DistilBERT V3 (Mean Pooling, batch=64, 15 epochs)")

## 5. Evaluate on 200 Test Samples

In [6]:
evaluate(runner.inference, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$3 $81 $12 $32 $23 $133 $68 $36 $8 $21 $37 $224 $21 $11 $2 $6 $47 $23 $15 $68 $76 $67 $37 $169 $80 $228 $198 $1 $48 $57 $40 $22 $67 $7 $6 $120 $73 $32 $32 $14 $109 $53 $41 $102 $107 $1 $3 $1 $73 $36 $6 $43 $249 $38 $23 $25 $13 $129 $59 $9 $127 $46 $24 $61 $355 $5 $49 $287 $20 $44 $16 $5 $83 $6 $15 $6 $57 $1 $3 $0 $8 $26 $6 $49 $14 $124 $115 $102 $31 $4 $37 $26 $0 $4 $2 $57 $9 $20 $4 $192 $5 $24 $4 $15 $6 $141 $1 $309 $16 $42 $22 $26 $5 $39 $19 $0 $16 $7 $64 $142 $7 $53 $12 $18 $20 $64 $1 $20 $67 $64 $54 $22 $6 $7 $52 $1 $48 $16 $31 $15 $1 $6 $0 $1 $21 $9 $39 $270 $57 $5 $3 $5 $2 $28 $40 $60 $16 $8 $91 $10 $80 $14 $7 $1 $282 $4 $81 $25 $1 $2 $65 $3 $201 $8 $24 $13 $11 $26 $58 $18 $129 $5 $76 $11 $10 $22 $86 $4 $15 $3 $3 $9 $11 $71 $3 $23 $32 $24 $15 $12 

## 6. Save Model Weights

In [7]:
runner.save("distilbert_model_v3.pth")
print("Saved to distilbert_model_v3.pth")

Saved to distilbert_model_v3.pth


## 7. Sanity Check

In [8]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
Actual:  $219.00
Predict: $222.34
Error:   $3.34
